# Download Data
## FloodNet NYC Tutorial
Author: Mark Bauer

# Summary
TBD

# Import Libraries

In [50]:
# libraries
import requests
import polars as pl
import geopandas as gpd

In [51]:
libraries = {
    "requests": requests.__version__,
    "polars": pl.__version__,
    "geopandas": gpd.__version__,
}

for name, version in libraries.items():
    print(f"{name:10s} {version}")

requests   2.32.5
polars     1.38.1
geopandas  1.1.2


# FloodNet Catalog on NYC Open Data

[FloodNet Catalog Link](https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=FloodNet+NYC&sortBy=relevance&page=1&pageSize=20)

# Data Dictionary and Dataset Description

In [52]:
## dat dictionary
# url on NYC Open Data
url = "https://data.cityofnewyork.us/api/views/kb2e-tjy3/files/\
9acd9a23-79a2-4ac0-929a-b0fa02182db6?download=true&filename=Flood%20Events_Data%20Dictionary.xlsx"

# saved file name
local_file = "data/Flood Events_Data Dictionary.xlsx"

# download file
response = requests.get(url)
response.raise_for_status()
with open(local_file, "wb") as f:
    f.write(response.content)

## dataset description
# url on NYC Open Data
url = "https://data.cityofnewyork.us/api/views/kb2e-tjy3/files/\
266f25e5-1809-40a0-afd5-50fb69596ce1?download=true&filename=Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf"

# saved file name
local_file = "data/Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf"

# download file
response = requests.get(url)
response.raise_for_status()
with open(local_file, "wb") as f:
    f.write(response.content)    

In [53]:
# sanity check
%ls data/

Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
metadata.csv


# Sensor Deployment Metadata

In [54]:
url = "https://data.cityofnewyork.us/resource/kb2e-tjy3.csv"
local_file = "data/metadata.csv"

# download file
response = requests.get(url)
response.raise_for_status()
with open(local_file, "wb") as f:
    f.write(response.content)

# sanity check file
metadata_df = pl.read_csv(local_file, try_parse_dates=True)

# inspect the shape of the data
n_rows, n_columns = metadata_df.shape
print("Metadata:")
print(f"Number of rows: {n_rows}")
print(f"Number of columns: {n_columns}")

Metadata:
Number of rows: 371
Number of columns: 16


In [55]:
# sanity check, preview data
metadata_df.head()

sensor_name,sensor_id,date_installed,tidally_influenced,date_removed,street_name,borough,zipcode,community_board,council_district,census_tract,nta,latitude,longitude,lowest_point_height_delta_inches,location
str,str,datetime[μs],str,datetime[μs],str,str,i64,i64,str,i64,str,f64,f64,f64,str
"""BK - Junius St/Sutter Ave""","""BK-junius-st-sutter-ave-1dnf80""",2023-05-24 00:00:00,"""No""",null,"""Junius Street""","""Brooklyn""",11212,316,"""BK16""",3090800,"""BK1602""",40.668742,-73.902658,13.43,"""POINT (-73.902658 40.668742)"""
"""BX - Metropolitan Ave/Linden D…","""BX-metropolitan-ave-linden-dr-…",2025-09-25 00:00:00,"""No""",null,"""Metropolitan Avenue""","""Bronx""",10462,209,"""BX09""",2021001,"""BX0904""",40.840601,-73.855459,8.78,"""POINT (-73.855459 40.840601)"""
"""Q - 147 Drive/ 259 St""","""Q-147-drive-259-st-1f5ig0""",2023-06-22 00:00:00,"""Yes""",2023-07-27 00:00:00,"""147th Drive""","""Queens""",11422,413,"""QN13""",4065600,"""QN1307""",40.65572,-73.729558,5.39,"""POINT (-73.729558 40.65572)"""
"""M - E 122nd St/3rd""","""M-e-122nd-st-3rd-1yjns0""",2022-11-13 00:00:00,"""No""",null,"""3rd Avenue""","""Manhattan""",10035,111,"""MN11""",1019600,"""MN1102""",40.801601,-73.937485,15.39,"""POINT (-73.937485 40.801601)"""
"""BX - Waters Pl/ Fink Ave""","""BX-roberts-ave-waters-pl-2921j…",2025-01-23 00:00:00,"""No""",null,"""Waters Place""","""Bronx""",10461,211,"""BX11""",2028400,"""BX1161""",40.843944,-73.839728,8.82,"""POINT (-73.839728 40.843944)"""


# Street Flooding Events Measured by FloodNet Sensors

In [56]:
# read in flood events as dataframe
max_limit = 500_000 # assign limit to large number
url = f"https://data.cityofnewyork.us/resource/aq7i-eu5q.csv?$limit={max_limit}"
local_file = "data/flood-events.csv"

# download file
response = requests.get(url)
response.raise_for_status()

with open(local_file, "wb") as f:
    f.write(response.content)

# sanity check
events_df = pl.read_csv(
    local_file,
    infer_schema_length=100000, # a few fiels have sparce data, so increase infer schema rows
    try_parse_dates=True # try to parse date fields
)

# inspect the shape of the data
n_rows, n_columns = events_df.shape
print("Flood Events:")
print(f"Number of rows: {n_rows}")
print(f"Number of columns: {n_columns}")

Flood Events:
Number of rows: 1887
Number of columns: 13


In [57]:
# sanity check, preview data
events_df.head()

sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,onset_time_mins,drain_time_mins,duration_mins,duration_above_4_inches_mins,duration_above_12_inches_mins,duration_above_24_inches_mins,flood_profile_depth_inches,flood_profile_time_secs
str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,str
"""Q - Beach 84 St""","""Q-beach-84-st-0me680""",2023-10-30 12:00:39,2023-10-30 16:38:19,17.72,133.86,143.81,277.66,211.41,103.6,0.0,"""[0.00, 0.87, 1.02, 1.38, 1.93,…","""[0, 1008, 1134, 1512, 1826, 20…"
"""BX - 1st St/Avenue A""","""BX-1st-st-avenue-a-1zby90""",2025-04-26 23:17:20,2025-04-27 11:55:34,6.18,27.95,730.29,758.24,132.81,0.0,0.0,"""[0.00, 0.71, 1.38, 1.38, 1.38,…","""[0, 64, 129, 132, 196, 260, 32…"
"""Q - Beach 35 St/Beach Channel …","""Q-beach-35-st-beach-channel-dr…",2024-02-09 11:41:01,2024-02-09 13:20:57,5.31,54.51,45.43,99.94,45.09,0.0,0.0,"""[0.00, 0.43, 0.63, 0.71, 0.94,…","""[0, 68, 136, 204, 273, 341, 40…"
"""BX - Tibbett Ave/W 234th St""","""BX-w-234th-st-tibbett-ave-2cak…",2025-08-20 21:05:19,2025-08-21 00:46:19,1.18,80.0,141.0,221.0,0.0,0.0,0.0,"""[0.00, 0.43, 0.47, 0.47, 0.51,…","""[0, 60, 120, 180, 240, 300, 36…"
"""Q - Davenport Ct 1""","""Q-davenport-ct-1-07zks0""",2024-01-10 00:39:25,2024-01-10 02:30:40,2.09,59.82,51.43,111.25,0.0,0.0,0.0,"""[0.00, 0.47, 0.47, 0.43, 0.43,…","""[0, 63, 127, 189, 252, 315, 37…"


# Flood Sensor Locations

In [58]:
# read in NYC Boroughs geometries from NYC Open Data
path = "https://data.cityofnewyork.us/resource/gthc-hcne.geojson"
boro_gdf = gpd.read_file(path)

# preview data, confirm CRS
print(boro_gdf.crs)
boro_gdf.head()

EPSG:4326


,borocode,boroname,shape_area,shape_leng,geometry
0,5,Staten Island,1623618358.46,325912.288988,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ..."
1,1,Manhattan,636631650.451,359537.866079,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ..."
2,2,Bronx,1187199300.36,463147.071867,"MULTIPOLYGON (((-73.89681 40.79581, -73.89694 ..."
3,3,Brooklyn,1934462607.43,726953.044632,"MULTIPOLYGON (((-73.86327 40.58388, -73.86381 ..."
4,4,Queens,3041419178.99,887905.076018,"MULTIPOLYGON (((-73.82645 40.59053, -73.82642 ..."


In [59]:
# save locally
boro_gdf.to_parquet('data/boroughs.parquet')

In [60]:
# sanity check, list files
%ls data/

Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
metadata.csv
